In [33]:
import pandas as pd
import requests
from dotenv import load_dotenv
import os
import time
from collections import Counter

def check_path(path):
    directory = os.path.dirname(path)
    if not os.path.exists(directory):
        print(f"Directory does not exist: {directory}")
        return False
    if not os.access(directory, os.W_OK):
        print(f"No write permission for: {directory}")
        return False
    print(f"OK, can write to: {path}")
    return True

def extract_canton(location_dict):
    if isinstance(location_dict, dict):
        area = location_dict.get("area", [])
        if len(area) > 1:
            return area[1]
    return None

def extract_skills(row):
    text = f"{row['title']} {row['description']}".lower()
    found = set()
    for keyword, canonical in SKILL_KEYWORDS.items():
        if keyword in text:
            found.add(canonical)
    return ", ".join(sorted(found)) if found else None



In [16]:
load_dotenv()
APP_ID = os.getenv("ADZUNA_APP_ID")
APP_KEY = os.getenv("ADZUNA_APP_KEY")

MAX_PAGES = 10
RESULTS_PER_PAGE = 50
MAX_RETRIES = 3

all_dfs = []

for page in range(1, MAX_PAGES + 1):

    for attempt in range(MAX_RETRIES):
        try:
            response = requests.get(
                f"https://api.adzuna.com/v1/api/jobs/ch/search/{page}",
                params={
                    "app_id": APP_ID,
                    "app_key": APP_KEY,
                    "results_per_page": RESULTS_PER_PAGE,
                    "what_or": "data analyst python automation intern werkstudent praktikant stagiaire business intelligence RPA"
                },
                timeout=10
            )
        except requests.exceptions.RequestException as e:
            print(f"Page {page}, attempt {attempt+1}: network error ({e})")
            time.sleep(5)
            continue

        if response.status_code == 200:
            break
        elif response.status_code == 429:
            print(f"Page {page}: rate limited, waiting 30s...")
            time.sleep(30)
        elif response.status_code in (401, 403):
            print("Auth error - check your API key. Stopping.")
            break
        else:
            print(f"Page {page}: unexpected status {response.status_code}, retrying...")
            time.sleep(5)
    else:
        print(f"Page {page}: failed after {MAX_RETRIES} attempts, stopping entirely.")
        break

    if response.status_code in (401, 403):
        break

    data = response.json()
    page_df = pd.DataFrame(data["results"])

    if page_df.empty:
        print(f"Page {page}: no more results, stopping.")
        break

    all_dfs.append(page_df)
    print(f"Page {page}: {len(page_df)} jobs collected")
    time.sleep(1)

df_all = pd.concat(all_dfs, ignore_index=True)
print(f"TOTAL: {len(df_all)} jobs collected")

if check_path("../data/raw/adzuna_jobs_raw.csv"):
    df_all.to_csv("../data/raw/adzuna_jobs_raw.csv", index=False)

Page 1: 50 jobs collected
Page 2: 50 jobs collected
Page 3: 50 jobs collected
Page 4: 50 jobs collected
Page 5: 50 jobs collected
Page 6: 50 jobs collected
Page 7: 50 jobs collected
Page 8: 50 jobs collected
Page 9: 50 jobs collected
Page 10: 50 jobs collected
TOTAL: 500 jobs collected
OK, can write to: ../data/raw/adzuna_jobs_raw.csv


In [17]:
print(df_all.shape)
print(df_all.isnull().sum())

(500, 17)
redirect_url             0
title                    0
category                 0
location                 0
longitude              122
created                  0
adref                    0
__CLASS__                0
latitude               122
salary_is_predicted      0
id                       0
description              0
company                  0
contract_type          476
contract_time          422
salary_max             486
salary_min             486
dtype: int64


In [18]:
print(df_all.duplicated(subset=["id"]).sum())

before = len(df_all)
df_all = df_all.drop_duplicates(subset=["id"], keep="first")
after = len(df_all)
print(f"Removed {before - after} duplicate rows")

0
Removed 0 duplicate rows


In [19]:
STUDENT_KEYWORDS = [
    "student", "intern", "internship",       # English
    "werkstudent", "praktikant", "praktikum", "lernende", "auszubildende",  # German
    "stagiaire", "alternance", "apprenti"     # French
]

pattern = "|".join(STUDENT_KEYWORDS)
df_all["is_student_role"] = df_all["title"].str.lower().str.contains(pattern, na=False)

print(df_all["is_student_role"].value_counts())

df_all["has_salary_info"] = df_all["salary_min"].notna()
print(df_all["has_salary_info"].value_counts())


is_student_role
False    346
True     154
Name: count, dtype: int64
has_salary_info
False    486
True      14
Name: count, dtype: int64


In [24]:
print(df_all.shape)
print(df_all.isnull().sum())
print(df_all.duplicated(subset=["id"]).sum())

before = len(df_all)
df_all = df_all.drop_duplicates(subset=["id"], keep="first")
print(f"Removed {before - len(df_all)} duplicate rows")

(500, 19)
redirect_url             0
title                    0
category                 0
location                 0
longitude              122
created                  0
adref                    0
__CLASS__                0
latitude               122
salary_is_predicted      0
id                       0
description              0
company                  0
contract_type          476
contract_time          422
salary_max             486
salary_min             486
is_student_role          0
has_salary_info          0
dtype: int64
0
Removed 0 duplicate rows


In [34]:
# --- Flatten nested columns ---
df_all["company_name"] = df_all["company"].apply(
    lambda x: x.get("display_name") if isinstance(x, dict) else None
)
df_all["location_name"] = df_all["location"].apply(
    lambda x: x.get("display_name") if isinstance(x, dict) else None
)
df_all["category_label"] = df_all["category"].apply(
    lambda x: x.get("label") if isinstance(x, dict) else None
)

# --- Extract canton from location ---
def extract_canton(location_dict):
    if isinstance(location_dict, dict):
        area = location_dict.get("area", [])
        if len(area) > 1:
            return area[1]
    return None

df_all["canton"] = df_all["location"].apply(extract_canton)
df_all["canton"] = df_all["canton"].fillna("Unknown")
df_all["category_label"] = df_all["category_label"].fillna("Unknown")

# print(df_all["canton"].value_counts(dropna=False))

df_all[["title", "company_name", "location_name", "canton", "category_label", "is_student_role", "has_salary_info"]].head()

# --- Extract mentioned skills from title + description ---
SKILL_KEYWORDS = {
    "python": "python",
    "sql": "sql",
    "excel": "excel",
    "power bi": "power bi",
    "powerbi": "power bi",
    "tableau": "tableau",
    "vba": "vba",
    "rpa": "rpa",
    "uipath": "rpa",
    "automation": "automation",
    "machine learning": "machine learning",
    "aws": "aws",
    "azure": "azure",
    "etl": "etl",
    "pandas": "pandas",
    "spark": "spark"
}


df_all["skills_mentioned"] = df_all.apply(extract_skills, axis=1)
print(f"Jobs with no matched skill: {df_all['skills_mentioned'].isna().sum()}")

all_skills = []
for skills_str in df_all["skills_mentioned"].dropna():
    all_skills.extend([s.strip() for s in skills_str.split(",")])

skill_counts = pd.Series(Counter(all_skills)).sort_values(ascending=False)
print(skill_counts.head(10))

Jobs with no matched skill: 375
automation          71
python              17
sql                 17
excel               17
machine learning     7
power bi             6
rpa                  3
tableau              2
etl                  2
spark                2
dtype: int64
